# Cycles experiment — Supplemental Figures 1 & 2

Wang et al. (NeurIPS 2024): 20 rooms × 30 cycles → 600 recorded rate-map trials.

- **Suppl. Fig. 1**: cross-correlation of population vectors (Pearson *r*), trials re-ordered by cycle (rooms sorted within each cycle).
- **Suppl. Fig. 2**: drift of six hidden units in **Room 1** across cycles.

Prerequisites: generate rooms under `data/cycles/` with `uv run generate-cycles-rooms`.

Training (no notebook overhead): `uv run cycles-experiment` (see `scripts/cycles_experiment.py`, config: `input_configs/all_cycles.json`).

Computation lives in `cycles_*.py`; this notebook configures training and builds the figures.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

for _path in (Path.cwd(), *Path.cwd().parents):
    if (_path / "pyproject.toml").is_file():
        _root = str(_path)
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break
else:
    raise RuntimeError("Could not find project root (pyproject.toml)")

import matplotlib.pyplot as plt

from cycles.cycles_paths import ROOMS_DIR, resolve_cycles_paths
from cycles.cycles_data import load_manifest
from cycles.cycles_train import CyclesConfig, load_cycles_result, run_cycles_experiment
from cycles.cycles_analysis import (
    pearson_correlation_matrix,
    population_vectors_from_ratemaps,
    ratemaps_for_room_across_cycles,
    reorder_trials_by_cycle,
    select_drift_cells,
)
from cycles.cycles_plots import (
    plot_suppl_fig1_correlation,
    plot_suppl_fig2_drift,
    save_figure,
)

manifest = load_manifest()
print(f"Room data: {ROOMS_DIR}")
print(f"Rooms in manifest: {manifest.n_rooms}, trial steps: {manifest.trial_steps}")


## Configuration

Set `SMOKE_TEST = True` for a quick run (2 rooms × 2 cycles). For paper figures use `SMOKE_TEST = False` (20 × 30; long on CPU).


In [ ]:
SMOKE_TEST = False  # False → full Suppl. Fig. 1/2 protocol
RUN_TRAINING = False  # False → only load saved results and plot
TRAIN_MODE = "indiv_traj"  # "default" | "indiv_traj"
TRAJECTORY_DURATION_S = 600.0  # null in JSON → 600 s
N_SEGMENTS = 8  # null → 4 (default) or 8 (indiv_traj)
STEP_SIZE = 20  # null → 20

if SMOKE_TEST:
    config = CyclesConfig(
        n_cycles=2,
        n_rooms=2,
        record_n_segments=50,
        train_mode=TRAIN_MODE,
        trajectory_duration_s=TRAJECTORY_DURATION_S,
        n_segments=N_SEGMENTS,
        step_size=STEP_SIZE,
    )
else:
    config = CyclesConfig(
        n_cycles=30,
        n_rooms=20,
        record_n_segments=None,
        train_mode=TRAIN_MODE,
        trajectory_duration_s=TRAJECTORY_DURATION_S,
        n_segments=N_SEGMENTS,
        step_size=STEP_SIZE,
    )

paths = resolve_cycles_paths(config)
paths.ckpt_dir.mkdir(parents=True, exist_ok=True)
paths.results_dir.mkdir(parents=True, exist_ok=True)
paths.plots_dir.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints: {paths.ckpt_dir}")
print(f"Results: {paths.results_dir}")
print(f"Plots: {paths.plots_dir}")

config


## Train across cycles and record rate maps


In [ ]:
SAVE_EVERY_K_CYCLES = 3

if RUN_TRAINING:
    result = run_cycles_experiment(
        config,
        results_dir=paths.results_dir,
        resume=True,
        save_every_k_cycles=SAVE_EVERY_K_CYCLES,
    )
else:
    LOAD_RESULTS = paths.results_dir / "cycles_ratemaps_truncated_10.npz"
    result = load_cycles_result(path=LOAD_RESULTS)

print("Rate maps:", result.ratemaps.shape)
print("Visits:", len(result.visit_indices))


## Supplemental Figure 1 — trial cross-correlation


In [ ]:
rm_sorted, cyc_sorted, room_sorted = reorder_trials_by_cycle(
    result.ratemaps,
    result.cycle_ids,
    result.room_ids,
)

pop_vecs = population_vectors_from_ratemaps(
    rm_sorted,
    room_size_cm=manifest.room_width_cm,
)
corr = pearson_correlation_matrix(pop_vecs)

fig1 = plot_suppl_fig1_correlation(
    corr,
    cycle_ids=cyc_sorted,
)
save_figure(fig1, paths.plots_dir / "suppl_fig1_correlation.pdf")
plt.show()


## Supplemental Figure 2 — drift in Room 1


In [ ]:
drift_room = 1  # room_01
rm_room, cycles_present = ratemaps_for_room_across_cycles(
    result.ratemaps,
    result.cycle_ids,
    result.room_ids,
    drift_room,
)

cell_ids = select_drift_cells(rm_room, n_cells=10, seed=0)
print("Selected units:", cell_ids)

fig2 = plot_suppl_fig2_drift(
    rm_room,
    cell_ids,
    cycles=cycles_present,
    room_label="Room 1",
)
save_figure(fig2, paths.plots_dir / "suppl_fig2_room1_drift.pdf")
plt.show()
